<a href="https://colab.research.google.com/github/Khaijiow/Managing-System/blob/main/SMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
from collections import Counter
import math
import itertools
import numpy as np

# เพิ่มไลบรารีสำหรับวาดกราฟ 3D
import plotly.graph_objects as go

print("โปรแกรมจัดเรียงสิ่งของในกล่องอัจฉริยะ (เวอร์ชันอัปเกรดการแสดงผล 3D)")

# ====== นิยามรูปทรงและสัญลักษณ์ ======
SHAPES = {
    "1x1": [(0, 0)], "2x2": [(0, 0), (0, 1), (1, 0), (1, 1)],
    "3x3": [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2),(2,0),(2,1),(2,2)],
    "1x2": [(0, 0), (0, 1)], "2x3": [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)],
    "3x4": [(0,0),(0,1),(0,2),(0,3),(1,0),(1,1),(1,2),(1,3),(2,0),(2,1),(2,2),(2,3)],
    "L": [(0, 0), (1, 0), (1, 1)], "T": [(0,0),(0,1),(0,2),(1,1)],
    "Z": [(0,0),(0,1),(1,1),(1,2)], "S": [(0,1),(0,2),(1,0),(1,1)]
}
SHAPE_LABELS = {
    "1x1": "1", "2x2": "2", "3x3": "3", "1x2": "4", "2x3": "5", "3x4": "6",
    "L": "L", "T": "T", "Z": "Z", "S": "S"
}
# กำหนดโทนสีสดใสให้วัตถุแต่ละชิ้นแยกแยะง่ายในโมเดล 3D
COLOR_MAP = {
    "1": "#FFD700", "2": "#FF4500", "3": "#1E90FF", "4": "#32CD32",
    "5": "#9370DB", "6": "#8B4513", "L": "#FF1493", "T": "#00CED1",
    "Z": "#008000", "S": "#FFA500", ".": "#E6E6E6"
}
SHAPE_VOLUMES = {name: len(coords) for name, coords in SHAPES.items()}

# ====== ฟังก์ชันสำหรับหมุนรูปทรง ======
def rotate_shape(coords):
    if not coords: return []
    max_r = max(r for r, c in coords)
    rotated = [(c, max_r - r) for r, c in coords]
    min_r, min_c = min(r for r,c in rotated), min(c for r,c in rotated)
    return sorted([(r - min_r, c - min_c) for r, c in rotated])

def get_all_rotations(coords):
    original_coords_tuple = tuple(coords)
    rotations, unique_coords_set = [], set()
    current_coords = coords
    for _ in range(4):
        current_coords_tuple = tuple(current_coords)
        if current_coords_tuple not in unique_coords_set:
            is_rotated = (current_coords_tuple != original_coords_tuple)
            rotations.append((current_coords, is_rotated))
            unique_coords_set.add(current_coords_tuple)
        current_coords = rotate_shape(current_coords)
    return rotations

ROTATED_SHAPES = {name: get_all_rotations(coords) for name, coords in SHAPES.items()}

# ====== ฟังก์ชัน Utility ======
def create_empty_grid(width, length, height):
    return [[[['.', False] for _ in range(width)] for _ in range(length)] for _ in range(height)]

def get_dimension_factors(volume, max_height=20):
    factor_triples = set()
    for h_candidate in range(1, int(volume** (1/3)) + 1):
        if volume % h_candidate == 0:
            area = volume // h_candidate
            for w_candidate in range(h_candidate, int(math.sqrt(area)) + 1):
                if area % w_candidate == 0:
                    l_candidate = area // w_candidate
                    factor_triples.add(tuple(sorted((l_candidate, w_candidate, h_candidate))))

    valid_dims = set()
    for triple in factor_triples:
        for p in set(itertools.permutations(triple)):
            if p[2] <= max_height:
                valid_dims.add(p)

    return sorted(list(valid_dims), key=lambda dim: abs(dim[0] - dim[1]))

# ====== ฟังก์ชันวางของ ======
def can_place(grid, shape_coords, h, r, c, box_dims, is_rotated):
    length, width, _ = box_dims
    for dr, dc in shape_coords:
        nr, nc = r + dr, c + dc
        if not (0 <= nr < length and 0 <= nc < width and grid[h][nr][nc][0] == '.'):
            return False
    if is_rotated:
        for dr, dc in shape_coords:
            nr, nc = r + dr, c + dc
            for move_r, move_c in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                adj_r, adj_c = nr + move_r, nc + move_c
                if not (0 <= adj_r < length and 0 <= adj_c < width): continue
                is_part_of_self = any((r + self_dr, c + self_dc) == (adj_r, adj_c) for self_dr, self_dc in shape_coords)
                if is_part_of_self: continue
                neighbor_label, neighbor_is_rotated = grid[h][adj_r][adj_c]
                if neighbor_label != '.' and neighbor_is_rotated: return False
    return True

def place_shape(grid, shape_coords, h, r_off, c_off, label_name, is_rotated):
    label_char = SHAPE_LABELS[label_name]
    for dr, dc in shape_coords:
        grid[h][r_off + dr][c_off + dc] = (label_char, is_rotated)

# ====== ฟังก์ชันจัดของหลัก ======
def pack_single_box(items_to_pack, box_dims):
    length, width, height = box_dims
    grid = create_empty_grid(width, length, height)
    items_counter = Counter(item[0] for item in items_to_pack)
    for h in range(height):
        for r in range(length):
            for c in range(width):
                if grid[h][r][c][0] == '.':
                    for shape_name in list(items_counter.keys()):
                        if items_counter[shape_name] > 0:
                            for rotated_coords, is_rotated in ROTATED_SHAPES[shape_name]:
                                if can_place(grid, rotated_coords, h, r, c, box_dims, is_rotated):
                                    place_shape(grid, rotated_coords, h, r, c, shape_name, is_rotated)
                                    items_counter[shape_name] -= 1
                                    goto_next_item = True
                                    break
                            else:
                                goto_next_item = False
                                continue
                            if goto_next_item: break
    unplaced_items = [item for name, count in items_counter.items() if count > 0 for item in [(name, 1)]*count]
    return grid, len(items_to_pack) > len(unplaced_items), unplaced_items

# ====== ✨ [ฟังก์ชันใหม่] แสดงผลแบบจำลองกล่องแบบ 3D Interactive ======
def visualize_box_3d(grid, box_dims, title="3D Box Visualization"):
    length, width, height = box_dims

    # พิกัด 3D ของลูกบาศก์ย่อยแต่ละบล็อก
    x_data, y_data, z_data = [], [], []
    colors = []
    hover_texts = []

    for h in range(height):
        for r in range(length):
            for c in range(width):
                label, is_rotated = grid[h][r][c]
                if label != '.':  # แสดงผลเฉพาะช่องที่มีของวางอยู่
                    x_data.append(c + 0.5)  # แกน X = กว้าง (Width)
                    y_data.append(r + 0.5)  # แกน Y = ยาว (Length)
                    z_data.append(h + 0.5)  # แกน Z = สูง (Height)
                    colors.append(COLOR_MAP.get(label, "#333333"))
                    rot_text = " (หมุน)" if is_rotated else ""
                    hover_texts.append(f"ชนิดวัตถุ: {label}{rot_text}<br>พิกัด: กว้าง={c}, ยาว={r}, ชั้น={h+1}")

    # สร้างวัตถุรูปทรง 3D Voxels (ลูกบาศก์)
    fig = go.Figure()

    if x_data:
        fig.add_trace(go.Scatter3d(
            x=x_data, y=y_data, z=z_data,
            mode='markers',
            marker=dict(
                size=18,
                color=colors,
                symbol='square',
                opacity=0.9,
                line=dict(color='black', width=1)
            ),
            text=hover_texts,
            hoverinfo='text'
        ))

    # ตั้งค่าตาราง ขอบเขตกล่อง และหน้าต่างการแสดงผล
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(title='ความกว้าง (Width)', range=[0, width], dtick=1),
            yaxis=dict(title='ความยาว (Length)', range=[0, length], dtick=1),
            zaxis=dict(title='ความสูง (Height/ชั้น)', range=[0, height], dtick=1),
            aspectmode='manual',
            aspectratio=dict(x=width, y=length, z=height)
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

# ฟังก์ชันแสดงผลตารางตัวอักษรแบบเดิม (เก็บไว้เป็นทางเลือกสำรอง)
def print_layer_packed(grid_layer):
    if not grid_layer: return
    length, width = len(grid_layer), len(grid_layer[0])
    for r in range(length):
        print("+" + "----+" * width)
        row_line = "|"
        for c in range(width):
            label, is_rotated = grid_layer[r][c]
            marker = "'" if is_rotated else " "
            row_line += f" {label}{marker} |"
        print(row_line)
    print("+" + "----+" * width)

def print_shapes_example():
    print("\nตัวอย่างสัญลักษณ์และรูปร่าง (ท่าเริ่มต้น):")
    for shape_name, coords in SHAPES.items():
        label = SHAPE_LABELS[shape_name]
        print(f"\n--- {shape_name} ({label}) ---")
        max_r, max_c = max(r for r,c in coords), max(c for r,c in coords)
        grid = [[' ' for _ in range(max_c+1)] for _ in range(max_r+1)]
        for r,c in coords: grid[r][c] = '#'
        for row in grid: print("".join(row))
    print("\nหมายเหตุ: ของที่ถูกหมุนจะมีเครื่องหมาย ' กำกับ และห้ามวางติดกัน")

def get_item_list():
    print_shapes_example()
    items = []
    for s in SHAPES.keys():
        while True:
            try:
                count_str = input(f"จำนวนชิ้น {s} (กด Enter ถ้าไม่เอา): ")
                if not count_str: break
                count = int(count_str)
                if count >= 0:
                    items.extend([(s, 1)] * count)
                    break
                else: print("จำนวนชิ้นต้องไม่ติดลบ")
            except ValueError: print("กรุณากรอกตัวเลข")
    print("\nสรุปชิ้นที่จะวางทั้งหมด:", ", ".join([f"{s}x{c}" for s,c in Counter(i[0] for i in items).items()]))
    return items

def find_smallest_box(items):
    total_volume = sum(SHAPE_VOLUMES[item[0]] for item in items)
    if total_volume == 0:
        print("\nไม่มีสิ่งของให้จัด ไม่จำเป็นต้องใช้กล่อง")
        return
    print(f"\nปริมาตรรวมของสิ่งของทั้งหมด: {total_volume} หน่วย")
    print("กำลังเริ่มค้นหากล่องขนาดเล็กที่สุด (สูงไม่เกิน 20, ฐานสมส่วน)...")

    current_volume = total_volume
    while True:
        print(f"\n--- กำลังทดสอบกล่องปริมาตร {current_volume} ---")
        possible_dims = get_dimension_factors(current_volume, max_height=20)
        if not possible_dims:
            print(f"ไม่มีขนาดกล่องที่สูงไม่เกิน 20 หน่วยสำหรับปริมาตรนี้, กำลังเพิ่มปริมาตร...")
            current_volume += 1
            continue

        for l, w, h in possible_dims:
            print(f" -> กำลังลองจัดลงกล่องขนาด {l}x{w}x{h} (ฐานต่าง {abs(l-w)})...")
            grid, _, unplaced = pack_single_box(items, (l, w, h))

            if not unplaced:
                print("\n" + "="*15 + " พบขนาดกล่องที่เหมาะสม! " + "="*15)
                print(f"ขนาดกล่องที่เล็กที่สุดที่พบ: ยาว {l}, กว้าง {w}, สูง {h}")
                print(f"ปริมาตรกล่อง: {l*w*h} หน่วย")
                print("="*52)

                print("\n[ระบบ] กำลังสร้างผังการจัดวางรูปแบบ 3D แบบโต้ตอบได้...")
                visualize_box_3d(grid, (l, w, h), title=f"กล่องขนาดเล็กที่สุด ({l}x{w}x{h})")
                return

        print(f"ไม่สามารถจัดของทั้งหมดลงในกล่องปริมาตร {current_volume} ได้, กำลังเพิ่มปริมาตร...")
        current_volume += 1

# ============ โปรแกรมหลัก ============
while True:
    main_mode = input("\nกรุณาเลือกโหมดการทำงาน:\n"
                      "  1. ค้นหากล่องขนาดเล็กที่สุดสำหรับของที่มี\n"
                      "  2. จัดของลงในกล่องที่กำหนดขนาดเอง\n"
                      "เลือกโหมด (1 หรือ 2): ")
    if main_mode in ['1', '2']: break
    print("ตัวเลือกไม่ถูกต้อง กรุณาเลือก 1 หรือ 2")

items = get_item_list()
if not items:
    print("\nไม่ได้ระบุสิ่งของ โปรแกรมจบการทำงาน")
else:
    if main_mode == '1':
        find_smallest_box(items)

    elif main_mode == '2':
        while True:
            try:
                WIDTH = int(input("\nกรอกความกว้างของกล่อง: "))
                LENGTH = int(input("กรอกความยาวของกล่อง: "))
                HEIGHT = int(input("กรอกความสูงของกล่อง: "))
                if WIDTH > 0 and LENGTH > 0 and HEIGHT > 0: break
                else: print("ขนาดต้องเป็นจำนวนเต็มบวก")
            except ValueError: print("กรุณากรอกตัวเลข")

        box_dims = (LENGTH, WIDTH, HEIGHT)
        box_count = 0
        remaining = list(items)

        if not remaining:
            print("\nไม่มีของให้จัด")
        else:
            print(f"\n[โหมด 2] กำลังคำนวณและแสดงผลการจัดเรียงลงกล่องขนาด {LENGTH}x{WIDTH}x{HEIGHT}...")

        while remaining:
            box_count += 1
            grid, has_placed, unplaced = pack_single_box(remaining, box_dims)

            if not has_placed:
                print(f"\n**ไม่สามารถจัดของที่เหลือ ({len(remaining)} ชิ้น) ลงในกล่องใบใหม่ได้อีก**")
                print("หยุดการจัดเรียง")
                break

            print(f"\n[ระบบ] กำลังเปิดแสดงผลลัพธ์โมเดล 3D สำหรับกล่องใบที่ {box_count}...")
            visualize_box_3d(grid, box_dims, title=f"รูปแบบการจัดเรียงกล่องใบที่ {box_count} ({LENGTH}x{WIDTH}x{HEIGHT})")

            remaining = unplaced

        print("\n" + "="*25)
        print("  การจัดเรียงเสร็จสิ้น!")
        print(f"  สรุป: ต้องใช้กล่องทั้งหมด {box_count} ใบ")
        if remaining:
             counts = Counter(item[0] for item in remaining)
             print("  และมีของเหลือที่จัดไม่ได้:", ", ".join([f"{s}: {c} ชิ้น" for s, c in counts.items()]))
        print("="*25)

print("\nโปรแกรมจบการทำงาน")


โปรแกรมจัดเรียงสิ่งของในกล่องอัจฉริยะ (เวอร์ชันอัปเกรดการแสดงผล 3D)

กรุณาเลือกโหมดการทำงาน:
  1. ค้นหากล่องขนาดเล็กที่สุดสำหรับของที่มี
  2. จัดของลงในกล่องที่กำหนดขนาดเอง
เลือกโหมด (1 หรือ 2): 1

ตัวอย่างสัญลักษณ์และรูปร่าง (ท่าเริ่มต้น):

--- 1x1 (1) ---
#

--- 2x2 (2) ---
##
##

--- 3x3 (3) ---
###
###
###

--- 1x2 (4) ---
##

--- 2x3 (5) ---
###
###

--- 3x4 (6) ---
####
####
####

--- L (L) ---
# 
##

--- T (T) ---
###
 # 

--- Z (Z) ---
## 
 ##

--- S (S) ---
 ##
## 

หมายเหตุ: ของที่ถูกหมุนจะมีเครื่องหมาย ' กำกับ และห้ามวางติดกัน
จำนวนชิ้น 1x1 (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น 2x2 (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น 3x3 (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น 1x2 (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น 2x3 (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น 3x4 (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น L (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น T (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น Z (กด Enter ถ้าไม่เอา): 10
จำนวนชิ้น S (กด Enter ถ้าไม่เอา): 10

สรุปชิ้นที่จะวางทั้งหมด: 1x1x10, 2x2x10, 3x3x10, 1x2x10, 2x3x10, 3x4x10, 


โปรแกรมจบการทำงาน


In [ ]:
2# -*- coding: utf-8 -*-
from collections import Counter
import math
import itertools
import numpy as np

import plotly.graph_objects as go

print("โปรแกรมจัดเรียงสิ่งของในกล่องอัจฉริยะ")

SHAPES = {
    "1x1": [(0, 0)], "2x2": [(0, 0), (0, 1), (1, 0), (1, 1)],
    "3x3": [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2),(2,0),(2,1),(2,2)],
    "1x2": [(0, 0), (0, 1)], "2x3": [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)],
    "3x4": [(0,0),(0,1),(0,2),(0,3),(1,0),(1,1),(1,2),(1,3),(2,0),(2,1),(2,2),(2,3)],
    "L": [(0, 0), (1, 0), (1, 1)], "T": [(0,0),(0,1),(0,2),(1,1)],
    "Z": [(0,0),(0,1),(1,1),(1,2)], "S": [(0,1),(0,2),(1,0),(1,1)]
}
SHAPE_LABELS = {
    "1x1": "1", "2x2": "2", "3x3": "3", "1x2": "4", "2x3": "5", "3x4": "6",
    "L": "L", "T": "T", "Z": "Z", "S": "S"
}
COLOR_MAP = {
    "1": "#FFD700", "2": "#FF4500", "3": "#1E90FF", "4": "#32CD32",
    "5": "#9370DB", "6": "#8B4513", "L": "#FF1493", "T": "#00CED1",
    "Z": "#008000", "S": "#FFA500", ".": "#E6E6E6"
}
SHAPE_VOLUMES = {name: len(coords) for name, coords in SHAPES.items()}

def rotate_shape(coords):
    if not coords: return []
    max_r = max(r for r, c in coords)
    rotated = [(c, max_r - r) for r, c in coords]
    min_r, min_c = min(r for r,c in rotated), min(c for r,c in rotated)
    return sorted([(r - min_r, c - min_c) for r, c in rotated])

def get_all_rotations(coords):
    original_coords_tuple = tuple(coords)
    rotations, unique_coords_set = [], set()
    current_coords = coords
    for _ in range(4):
        current_coords_tuple = tuple(current_coords)
        if current_coords_tuple not in unique_coords_set:
            is_rotated = (current_coords_tuple != original_coords_tuple)
            rotations.append((current_coords, is_rotated))
            unique_coords_set.add(current_coords_tuple)
        current_coords = rotate_shape(current_coords)
    return rotations

ROTATED_SHAPES = {name: get_all_rotations(coords) for name, coords in SHAPES.items()}

def create_empty_grid(width, length, height):
    return [[[['.', False] for _ in range(width)] for _ in range(length)] for _ in range(height)]

def get_dimension_factors(volume, max_height=20):
    factor_triples = set()
    for h_candidate in range(1, int(volume** (1/3)) + 1):
        if volume % h_candidate == 0:
            area = volume // h_candidate
            for w_candidate in range(h_candidate, int(math.sqrt(area)) + 1):
                if area % w_candidate == 0:
                    l_candidate = area // w_candidate
                    factor_triples.add(tuple(sorted((l_candidate, w_candidate, h_candidate))))

    valid_dims = set()
    for triple in factor_triples:
        for p in set(itertools.permutations(triple)):
            if p[2] <= max_height:
                valid_dims.add(p)

    return sorted(list(valid_dims), key=lambda dim: abs(dim[0] - dim[1]))

def can_place(grid, shape_coords, h, r, c, box_dims, is_rotated):
    length, width, _ = box_dims
    for dr, dc in shape_coords:
        nr, nc = r + dr, c + dc
        if not (0 <= nr < length and 0 <= nc < width and grid[h][nr][nc][0] == '.'):
            return False
    if is_rotated:
        for dr, dc in shape_coords:
            nr, nc = r + dr, c + dc
            for move_r, move_c in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                adj_r, adj_c = nr + move_r, nc + move_c
                if not (0 <= adj_r < length and 0 <= adj_c < width): continue
                is_part_of_self = any((r + self_dr, c + self_dc) == (adj_r, adj_c) for self_dr, self_dc in shape_coords)
                if is_part_of_self: continue
                neighbor_label, neighbor_is_rotated = grid[h][adj_r][adj_c]
                if neighbor_label != '.' and neighbor_is_rotated: return False
    return True

def place_shape(grid, shape_coords, h, r_off, c_off, label_name, is_rotated):
    label_char = SHAPE_LABELS[label_name]
    for dr, dc in shape_coords:
        grid[h][r_off + dr][c_off + dc] = (label_char, is_rotated)

def pack_single_box(items_to_pack, box_dims):
    length, width, height = box_dims
    grid = create_empty_grid(width, length, height)
    items_counter = Counter(item[0] for item in items_to_pack)
    for h in range(height):
        for r in range(length):
            for c in range(width):
                if grid[h][r][c][0] == '.':
                    for shape_name in list(items_counter.keys()):
                        if items_counter[shape_name] > 0:
                            for rotated_coords, is_rotated in ROTATED_SHAPES[shape_name]:
                                if can_place(grid, rotated_coords, h, r, c, box_dims, is_rotated):
                                    place_shape(grid, rotated_coords, h, r, c, shape_name, is_rotated)
                                    items_counter[shape_name] -= 1
                                    goto_next_item = True
                                    break
                            else:
                                goto_next_item = False
                                continue
                            if goto_next_item: break
    unplaced_items = [item for name, count in items_counter.items() if count > 0 for item in [(name, 1)]*count]
    return grid, len(items_to_pack) > len(unplaced_items), unplaced_items

def visualize_box_3d(grid, box_dims, title="3D Box Visualization"):
    length, width, height = box_dims

    # พิกัด 3D ของลูกบาศก์ย่อยแต่ละบล็อก
    x_data, y_data, z_data = [], [], []
    colors = []
    hover_texts = []

    for h in range(height):
        for r in range(length):
            for c in range(width):
                label, is_rotated = grid[h][r][c]
                if label != '.':  # แสดงผลเฉพาะช่องที่มีของวางอยู่
                    x_data.append(c + 0.5)  # แกน X = กว้าง (Width)
                    y_data.append(r + 0.5)  # แกน Y = ยาว (Length)
                    z_data.append(h + 0.5)  # แกน Z = สูง (Height)
                    colors.append(COLOR_MAP.get(label, "#333333"))
                    rot_text = " (หมุน)" if is_rotated else ""
                    hover_texts.append(f"ชนิดวัตถุ: {label}{rot_text}<br>พิกัด: กว้าง={c}, ยาว={r}, ชั้น={h+1}")

    fig = go.Figure()

    if x_data:
        fig.add_trace(go.Scatter3d(
            x=x_data, y=y_data, z=z_data,
            mode='markers',
            marker=dict(
                size=18,
                color=colors,
                symbol='square',
                opacity=0.9,
                line=dict(color='black', width=1)
            ),
            text=hover_texts,
            hoverinfo='text'
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(title='ความกว้าง (Width)', range=[0, width], dtick=1),
            yaxis=dict(title='ความยาว (Length)', range=[0, length], dtick=1),
            zaxis=dict(title='ความสูง (Height/ชั้น)', range=[0, height], dtick=1),
            aspectmode='manual',
            aspectratio=dict(x=width, y=length, z=height)
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

def print_layer_packed(grid_layer):
    if not grid_layer: return
    length, width = len(grid_layer), len(grid_layer[0])
    for r in range(length):
        print("+" + "----+" * width)
        row_line = "|"
        for c in range(width):
            label, is_rotated = grid_layer[r][c]
            marker = "'" if is_rotated else " "
            row_line += f" {label}{marker} |"
        print(row_line)
    print("+" + "----+" * width)

def print_shapes_example():
    print("\nตัวอย่างสัญลักษณ์และรูปร่าง (ท่าเริ่มต้น):")
    for shape_name, coords in SHAPES.items():
        label = SHAPE_LABELS[shape_name]
        print(f"\n--- {shape_name} ({label}) ---")
        max_r, max_c = max(r for r,c in coords), max(c for r,c in coords)
        grid = [[' ' for _ in range(max_c+1)] for _ in range(max_r+1)]
        for r,c in coords: grid[r][c] = '#'
        for row in grid: print("".join(row))
    print("\nหมายเหตุ: ของที่ถูกหมุนจะมีเครื่องหมาย ' กำกับ และห้ามวางติดกัน")

def get_item_list():
    print_shapes_example()
    items = []
    for s in SHAPES.keys():
        while True:
            try:
                count_str = input(f"จำนวนชิ้น {s} (กด Enter ถ้าไม่เอา): ")
                if not count_str: break
                count = int(count_str)
                if count >= 0:
                    items.extend([(s, 1)] * count)
                    break
                else: print("จำนวนชิ้นต้องไม่ติดลบ")
            except ValueError: print("กรุณากรอกตัวเลข")
    print("\nสรุปชิ้นที่จะวางทั้งหมด:", ", ".join([f"{s}x{c}" for s,c in Counter(i[0] for i in items).items()]))
    return items

def find_smallest_box(items):
    total_volume = sum(SHAPE_VOLUMES[item[0]] for item in items)
    if total_volume == 0:
        print("\nไม่มีสิ่งของให้จัด ไม่จำเป็นต้องใช้กล่อง")
        return
    print(f"\nปริมาตรรวมของสิ่งของทั้งหมด: {total_volume} หน่วย")
    print("กำลังเริ่มค้นหากล่องขนาดเล็กที่สุด (สูงไม่เกิน 20, ฐานสมส่วน)...")

    current_volume = total_volume
    while True:
        print(f"\n--- กำลังทดสอบกล่องปริมาตร {current_volume} ---")
        possible_dims = get_dimension_factors(current_volume, max_height=20)
        if not possible_dims:
            print(f"ไม่มีขนาดกล่องที่สูงไม่เกิน 20 หน่วยสำหรับปริมาตรนี้, กำลังเพิ่มปริมาตร...")
            current_volume += 1
            continue

        for l, w, h in possible_dims:
            print(f" -> กำลังลองจัดลงกล่องขนาด {l}x{w}x{h} (ฐานต่าง {abs(l-w)})...")
            grid, _, unplaced = pack_single_box(items, (l, w, h))

            if not unplaced:
                print("\n" + "="*15 + " พบขนาดกล่องที่เหมาะสม! " + "="*15)
                print(f"ขนาดกล่องที่เล็กที่สุดที่พบ: ยาว {l}, กว้าง {w}, สูง {h}")
                print(f"ปริมาตรกล่อง: {l*w*h} หน่วย")
                print("="*52)

                print("\n[ระบบ] กำลังสร้างผังการจัดวางรูปแบบ 3D แบบโต้ตอบได้...")
                visualize_box_3d(grid, (l, w, h), title=f"กล่องขนาดเล็กที่สุด ({l}x{w}x{h})")
                return

        print(f"ไม่สามารถจัดของทั้งหมดลงในกล่องปริมาตร {current_volume} ได้, กำลังเพิ่มปริมาตร...")
        current_volume += 1

while True:
    main_mode = input("\nกรุณาเลือกโหมดการทำงาน:\n"
                      "  1. ค้นหากล่องขนาดเล็กที่สุดสำหรับของที่มี\n"
                      "  2. จัดของลงในกล่องที่กำหนดขนาดเอง\n"
                      "เลือกโหมด (1 หรือ 2): ")
    if main_mode in ['1', '2']: break
    print("ตัวเลือกไม่ถูกต้อง กรุณาเลือก 1 หรือ 2")

items = get_item_list()
if not items:
    print("\nไม่ได้ระบุสิ่งของ โปรแกรมจบการทำงาน")
else:
    if main_mode == '1':
        find_smallest_box(items)

    elif main_mode == '2':
        while True:
            try:
                WIDTH = int(input("\nกรอกความกว้างของกล่อง: "))
                LENGTH = int(input("กรอกความยาวของกล่อง: "))
                HEIGHT = int(input("กรอกความสูงของกล่อง: "))
                if WIDTH > 0 and LENGTH > 0 and HEIGHT > 0: break
                else: print("ขนาดต้องเป็นจำนวนเต็มบวก")
            except ValueError: print("กรุณากรอกตัวเลข")

        box_dims = (LENGTH, WIDTH, HEIGHT)
        box_count = 0
        remaining = list(items)

        if not remaining:
            print("\nไม่มีของให้จัด")
        else:
            print(f"\n[โหมด 2] กำลังคำนวณและแสดงผลการจัดเรียงลงกล่องขนาด {LENGTH}x{WIDTH}x{HEIGHT}...")

        while remaining:
            box_count += 1
            grid, has_placed, unplaced = pack_single_box(remaining, box_dims)

            if not has_placed:
                print(f"\n**ไม่สามารถจัดของที่เหลือ ({len(remaining)} ชิ้น) ลงในกล่องใบใหม่ได้อีก**")
                print("หยุดการจัดเรียง")
                break

            print(f"\n[ระบบ] กำลังเปิดแสดงผลลัพธ์โมเดล 3D สำหรับกล่องใบที่ {box_count}...")
            visualize_box_3d(grid, box_dims, title=f"รูปแบบการจัดเรียงกล่องใบที่ {box_count} ({LENGTH}x{WIDTH}x{HEIGHT})")

            remaining = unplaced

        print("\n" + "="*25)
        print("  การจัดเรียงเสร็จสิ้น!")
        print(f"  สรุป: ต้องใช้กล่องทั้งหมด {box_count} ใบ")
        if remaining:
             counts = Counter(item[0] for item in remaining)
             print("  และมีของเหลือที่จัด1ไม่ได้:", ", ".join([f"{s}: {c} ชิ้น" for s, c in counts.items()]))
        print("="*25)

print("\nโปรแกรมจบการทำงาน")
1

โปรแกรมจัดเรียงสิ่งของในกล่องอัจฉริยะ
